# Raw Uni Enrollments AUS

Notebook for exploring enrollment and funding data.

In [1]:
import os
import pandas as pd
from IPython.display import display

paths_to_try = [
    os.path.join('..', 'data', 'raw', 'EnrollmentsAUS_category.csv'),
    os.path.join('data', 'raw', 'EnrollmentsAUS_category.csv'),
]

existing_path = next((path for path in paths_to_try if os.path.exists(path)), None)
if existing_path is None:
    raise FileNotFoundError(f"Could not find the raw enrollments category file. Checked: {paths_to_try}")

raw_uni_enrollments_df = pd.read_csv(existing_path)
print(f"Loaded file: {existing_path}")
display(raw_uni_enrollments_df)

Loaded file: ..\data\raw\EnrollmentsAUS_category.csv


,Category,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Natural & Physical Science,119102,124209,129609,134334,136733,139629,134424,132406,134511
1,Information Technology,66355,80090,99986,116623,120916,116296,119278,143253,164464
2,Engineering & Related Tech,111060,115231,119909,121980,118229,112931,109812,115197,122954
3,Architecture & Building,32074,35604,39459,42370,43478,43773,42327,42174,40942
4,Environment & Related,18537,18260,18055,18747,21916,20776,19476,19284,19376
5,Health,235033,247198,256324,265586,277590,288385,280887,283060,295498
6,Education,128277,128603,126360,127202,135250,141971,136096,137387,153177
7,Management & Commerce,380800,389836,396813,399609,380050,352882,340454,360236,368989
8,Society & Culture,312569,322471,326071,332155,343667,350654,334897,326512,329590
9,Creative Arts,93916,94955,96195,97478,97745,98528,93412,92496,96142


In [2]:
import os
import pandas as pd
from IPython.display import display

paths_to_try = [
    os.path.join('..', 'data', 'raw', 'EnrollmentsAUS_category_with_key.csv'),
    os.path.join('data', 'raw', 'EnrollmentsAUS_category_with_key.csv'),
]

existing_path = next((path for path in paths_to_try if os.path.exists(path)), None)
if existing_path is None:
    raise FileNotFoundError(f"Could not find the category-with-key file. Checked: {paths_to_try}")

category_with_key_df = pd.read_csv(existing_path)
print(f"Loaded file: {existing_path}")
print(f"Rows: {len(category_with_key_df)}, Columns: {len(category_with_key_df.columns)}")
display(category_with_key_df)

Loaded file: ..\data\raw\EnrollmentsAUS_category_with_key.csv
Rows: 12, Columns: 11


,category_key,Category,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,CAT001,Natural & Physical Science,119102,124209,129609,134334,136733,139629,134424,132406,134511
1,CAT002,Information Technology,66355,80090,99986,116623,120916,116296,119278,143253,164464
2,CAT003,Engineering & Related Tech,111060,115231,119909,121980,118229,112931,109812,115197,122954
3,CAT004,Architecture & Building,32074,35604,39459,42370,43478,43773,42327,42174,40942
4,CAT005,Environment & Related,18537,18260,18055,18747,21916,20776,19476,19284,19376
5,CAT006,Health,235033,247198,256324,265586,277590,288385,280887,283060,295498
6,CAT007,Education,128277,128603,126360,127202,135250,141971,136096,137387,153177
7,CAT008,Management & Commerce,380800,389836,396813,399609,380050,352882,340454,360236,368989
8,CAT009,Society & Culture,312569,322471,326071,332155,343667,350654,334897,326512,329590
9,CAT010,Creative Arts,93916,94955,96195,97478,97745,98528,93412,92496,96142


In [3]:
# Add category keys to AUS uni enrollments table

import re



def normalize_category_text(value):

    text = str(value).strip().lower()

    text = re.sub(r"\s*&\s*", " and ", text)

    text = re.sub(r"[^a-z0-9]+", " ", text)

    return re.sub(r"\s+", " ", text).strip()



raw_df = raw_uni_enrollments_df.copy()

key_df = category_with_key_df.copy()



raw_category_col = next((c for c in raw_df.columns if str(c).strip().lower() == "category"), None)

key_category_col = next((c for c in key_df.columns if str(c).strip().lower() == "category"), None)

key_key_col = next((c for c in key_df.columns if "key" in str(c).strip().lower()), None)



if raw_category_col is None or key_category_col is None or key_key_col is None:

    raise ValueError("Could not detect required columns: Category and category_key")



key_lookup = key_df[[key_category_col, key_key_col]].dropna().drop_duplicates().copy()

key_lookup["_category_norm"] = key_lookup[key_category_col].map(normalize_category_text)

key_lookup = key_lookup[["_category_norm", key_key_col]].drop_duplicates(subset=["_category_norm"])



raw_df["_category_norm"] = raw_df[raw_category_col].map(normalize_category_text)

raw_uni_enrollments_with_key_df = raw_df.merge(key_lookup, on="_category_norm", how="left")



raw_uni_enrollments_with_key_df = raw_uni_enrollments_with_key_df.rename(columns={key_key_col: "category_key"})

raw_uni_enrollments_with_key_df = raw_uni_enrollments_with_key_df.drop(columns=["_category_norm"])



unmatched = raw_uni_enrollments_with_key_df[raw_uni_enrollments_with_key_df["category_key"].isna()]

print(f"Rows with category_key added: {len(raw_uni_enrollments_with_key_df)}")

print(f"Unmatched categories: {len(unmatched)}")

if len(unmatched) > 0:

    display(unmatched[[raw_category_col]].drop_duplicates())



display(raw_uni_enrollments_with_key_df)


Rows with category_key added: 12
Unmatched categories: 0


,Category,2016,2017,2018,2019,2020,2021,2022,2023,2024,category_key
0,Natural & Physical Science,119102,124209,129609,134334,136733,139629,134424,132406,134511,CAT001
1,Information Technology,66355,80090,99986,116623,120916,116296,119278,143253,164464,CAT002
2,Engineering & Related Tech,111060,115231,119909,121980,118229,112931,109812,115197,122954,CAT003
3,Architecture & Building,32074,35604,39459,42370,43478,43773,42327,42174,40942,CAT004
4,Environment & Related,18537,18260,18055,18747,21916,20776,19476,19284,19376,CAT005
5,Health,235033,247198,256324,265586,277590,288385,280887,283060,295498,CAT006
6,Education,128277,128603,126360,127202,135250,141971,136096,137387,153177,CAT007
7,Management & Commerce,380800,389836,396813,399609,380050,352882,340454,360236,368989,CAT008
8,Society & Culture,312569,322471,326071,332155,343667,350654,334897,326512,329590,CAT009
9,Creative Arts,93916,94955,96195,97478,97745,98528,93412,92496,96142,CAT010


In [4]:
# Apply numeric CategoryKey values to enrollment data (aligned with funding categories)

import re



def normalize_category(value):

    text = str(value).strip().lower()

    text = re.sub(r"\s*&\s*", " and ", text)

    text = re.sub(r"[^a-z0-9]+", " ", text)

    return re.sub(r"\s+", " ", text).strip()



categorykey_map = {

    "natural and physical science": 1,

    "information technology": 2,

    "engineering and related tech": 3,

    "architecture and building": 4,

    "environment and related": 5,

    "health": 6,

    "education": 7,

    "management and commerce": 8,

    "society and culture": 9,

    "creative arts": 10,

    "others": 11,

    "total": 99,

}



enrollment_with_categorykey_df = raw_uni_enrollments_df.copy()

enrollment_with_categorykey_df["CategoryKey"] = enrollment_with_categorykey_df["Category"].map(

    lambda x: categorykey_map.get(normalize_category(x))

)



missing = enrollment_with_categorykey_df[enrollment_with_categorykey_df["CategoryKey"].isna()]

print(f"Rows: {len(enrollment_with_categorykey_df)}")

print(f"Missing CategoryKey rows: {len(missing)}")

if len(missing) > 0:

    display(missing[["Category"]])



display(enrollment_with_categorykey_df)


Rows: 12
Missing CategoryKey rows: 0


,Category,2016,2017,2018,2019,2020,2021,2022,2023,2024,CategoryKey
0,Natural & Physical Science,119102,124209,129609,134334,136733,139629,134424,132406,134511,1
1,Information Technology,66355,80090,99986,116623,120916,116296,119278,143253,164464,2
2,Engineering & Related Tech,111060,115231,119909,121980,118229,112931,109812,115197,122954,3
3,Architecture & Building,32074,35604,39459,42370,43478,43773,42327,42174,40942,4
4,Environment & Related,18537,18260,18055,18747,21916,20776,19476,19284,19376,5
5,Health,235033,247198,256324,265586,277590,288385,280887,283060,295498,6
6,Education,128277,128603,126360,127202,135250,141971,136096,137387,153177,7
7,Management & Commerce,380800,389836,396813,399609,380050,352882,340454,360236,368989,8
8,Society & Culture,312569,322471,326071,332155,343667,350654,334897,326512,329590,9
9,Creative Arts,93916,94955,96195,97478,97745,98528,93412,92496,96142,10


In [5]:
# Export enrollment data with numeric CategoryKey (separate clean output)

from pathlib import Path



if "enrollment_with_categorykey_df" not in globals():

    raise ValueError("Run Cell 5 first to create enrollment_with_categorykey_df.")



candidate_dirs = [

    Path.cwd() / "data" / "clean",

    Path.cwd().parent / "data" / "clean",

]

output_dir = next((path for path in candidate_dirs if path.parent.exists()), candidate_dirs[0])

output_dir.mkdir(parents=True, exist_ok=True)



output_path = output_dir / "EnrollmentsAUS_category_with_numeric_key.csv"

enrollment_with_categorykey_df.to_csv(output_path, index=False)



print(f"Exported: {output_path}")

print(f"Rows: {len(enrollment_with_categorykey_df)}, Columns: {len(enrollment_with_categorykey_df.columns)}")

display(enrollment_with_categorykey_df)


Exported: c:\Users\neddp\ECC3479-Project-JRGS\reproduction test\data\clean\EnrollmentsAUS_category_with_numeric_key.csv
Rows: 12, Columns: 11


,Category,2016,2017,2018,2019,2020,2021,2022,2023,2024,CategoryKey
0,Natural & Physical Science,119102,124209,129609,134334,136733,139629,134424,132406,134511,1
1,Information Technology,66355,80090,99986,116623,120916,116296,119278,143253,164464,2
2,Engineering & Related Tech,111060,115231,119909,121980,118229,112931,109812,115197,122954,3
3,Architecture & Building,32074,35604,39459,42370,43478,43773,42327,42174,40942,4
4,Environment & Related,18537,18260,18055,18747,21916,20776,19476,19284,19376,5
5,Health,235033,247198,256324,265586,277590,288385,280887,283060,295498,6
6,Education,128277,128603,126360,127202,135250,141971,136096,137387,153177,7
7,Management & Commerce,380800,389836,396813,399609,380050,352882,340454,360236,368989,8
8,Society & Culture,312569,322471,326071,332155,343667,350654,334897,326512,329590,9
9,Creative Arts,93916,94955,96195,97478,97745,98528,93412,92496,96142,10
